In [1]:
# Load environment variables and verify the project setup.
import sys, types
from pathlib import Path

# --- RAGAS compatibility shim (must run before importing ragas anywhere) ---
# ragas 0.4.x does `from langchain_community.chat_models.vertexai import ChatVertexAI`,
# a submodule removed in langchain-community 0.4.x. ragas only imports the NAME (never
# instantiates it unless you use Vertex AI), so register a stub to satisfy the import —
# this avoids pulling the heavy langchain-google-vertexai package.
if "langchain_community.chat_models.vertexai" not in sys.modules:
    _shim = types.ModuleType("langchain_community.chat_models.vertexai")
    _shim.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _shim

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-classic  — installed (1.0.7)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-experimental  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ragas  — installed (0.4.3)
  ✓ rank-bm25  — installed (0.2.2)
  ✓ rapidfuzz  — installed (3.14.5)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Context Precision — with examples

**Context precision** measures whether the *relevant* retrieved contexts are ranked **near the top**. Below we build a real RAG pipeline (load → semantic chunks → Chroma → LCEL answer), then score the retrieved contexts with RAGAS.

> ragas 0.4.x imports a `langchain_community` Vertex AI submodule removed in LangChain 1.x; the setup cell above registers a small stub to satisfy that import.

## End-to-end RAG over the example PDF

Generate a *real* answer to evaluate: load the SDLC PDF → semantic chunks → store embeddings in Chroma → LCEL chain to query & answer. The faithfulness section below can then score this generated `rag_answer` against `rag_context`.

### 1. Load the PDF

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"
docs = PyPDFLoader(str(pdf_path)).load()
print(f"Loaded {len(docs)} page(s)")

/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_76204/4130999730.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 12 page(s)


### 2. Semantic chunking

In [3]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
chunks = SemanticChunker(
    embedding_model, breakpoint_threshold_type="percentile"
).split_documents(docs)
print(f"{len(docs)} pages -> {len(chunks)} semantic chunks")

/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_76204/1029204723.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


12 pages -> 23 semantic chunks


### 3. Embed & store in Chroma

In [4]:
import os
from langchain_chroma import Chroma

persist_dir = str(ROOT / os.getenv("CHROMA_PERSIST_DIR", "./chroma_db"))
vector_store = Chroma(
    collection_name="faithfulness_demo",
    embedding_function=embedding_model,
    persist_directory=persist_dir,
)
vector_store.reset_collection()        # start clean so re-runs do not duplicate
vector_store.add_documents(chunks)
print("Stored", vector_store._collection.count(), "chunks in collection 'faithfulness_demo'")

Stored 23 chunks in collection 'faithfulness_demo'


### 4. LCEL chain — query & answer

`{context, question} | prompt | llm | parser`. We also capture the retrieved `rag_context` separately so faithfulness can check the answer against it.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

question = "What is Low-Level Design (LLD) and who is responsible for it?"
rag_context = format_docs(retriever.invoke(question))   # context the answer is grounded in
rag_answer = rag_chain.invoke(question)

print("Q:", question)
print("A:", rag_answer)

Q: What is Low-Level Design (LLD) and who is responsible for it?
A: Low-Level Design (LLD) provides detailed internal design of each component, serving as the blueprint from which developers code. It includes class/module structure, methods and responsibilities, detailed algorithms, data structures, database tables, API endpoint signatures, and more. The owner of LLD is the Senior Developers / Tech Lead.


## Context precision with RAGAS

`LLMContextPrecisionWithoutReference` uses the generated `rag_answer` (no hand-written reference needed): for each retrieved context it judges whether that context was useful for producing the answer, then computes the rank-weighted precision. Score near 1.0 means relevant contexts are ranked first.

In [7]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
context_precision = LLMContextPrecisionWithoutReference(llm=eval_llm)

retrieved = retriever.invoke(question)
sample = SingleTurnSample(
    user_input=question,
    response=rag_answer,
    retrieved_contexts=[d.page_content for d in retrieved],
)

# Top-level await works in the Jupyter kernel.
score = await context_precision.single_turn_ascore(sample)
print(f"Retrieved {len(retrieved)} contexts")
print(f"RAGAS context precision = {score:.3f}")


/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_76204/2288322955.py:2: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import LLMContextPrecisionWithoutReference
/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_76204/2288322955.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))


Retrieved 4 contexts
RAGAS context precision = 1.000
['SDLC — End-to-End Reference\nPage 5\n3. Design\nDefines how the system will be built — architecture, components, interfaces, data, and quality\nattributes. 3.1 High-Level Design (HLD) / System Architecture\nPurpose. Describes the overall system architecture, major components and how they interact. Owner: Solution Architect  |  Input: SRS, TRD  |  Flows to: LLD, infra setup\n\x7f\nSystem architecture diagram and major subsystems/services\n\x7f\nComponent responsibilities and interactions\n\x7f\nTechnology choices, data flow and integration overview\n\x7f\nDeployment topology and scalability/availability approach\n\x7f\nCross-cutting concerns: security, logging, caching, resilience\n3.2 Low-Level Design (LLD)\nPurpose. Provides detailed internal design of each component — the blueprint developers code from. Owner: Senior Developers / Tech Lead  |  Input: HLD  |  Flows to: Implementation\n\x7f\nClass/module structure, methods and resp

In [8]:
for d in retrieved:
    print("=" * 80)
    print(d.metadata)
    print(d.page_content)
    

{'total_pages': 12, 'source': '/Users/Prabhukumar/Projects/PycharmProjects/rag-reference/assets/sample-docs/sdlc-end-to-end.pdf', 'page_label': '5', 'creator': '(unspecified)', 'trapped': '/False', 'author': 'Prabhukumar Sivamoorthy (Prabhukumarsivamoorthy@gmail.com)', 'page': 4, 'producer': 'ReportLab PDF Library - (opensource)', 'creationdate': '2026-05-30T21:30:41-07:00', 'moddate': '2026-05-30T21:30:41-07:00', 'subject': '(unspecified)', 'title': 'SDLC End-to-End Reference', 'keywords': ''}
SDLC — End-to-End Reference
Page 5
3. Design
Defines how the system will be built — architecture, components, interfaces, data, and quality
attributes. 3.1 High-Level Design (HLD) / System Architecture
Purpose. Describes the overall system architecture, major components and how they interact. Owner: Solution Architect  |  Input: SRS, TRD  |  Flows to: LLD, infra setup

System architecture diagram and major subsystems/services

Component responsibilities and interactions

Technology choices, d

### Variant: with a reference answer

`LLMContextPrecisionWithReference` judges each context against a known-good `reference` answer instead of the model's response — useful when you have a labelled test set.

In [7]:
from ragas.metrics import LLMContextPrecisionWithReference

reference = ("Low-Level Design (LLD) is the detailed internal design of each component, "
             "owned by the senior developers / tech lead.")

cp_ref = LLMContextPrecisionWithReference(llm=eval_llm)
sample_ref = SingleTurnSample(
    user_input=question,
    reference=reference,
    retrieved_contexts=[d.page_content for d in retrieved],
)
score_ref = await cp_ref.single_turn_ascore(sample_ref)
print(f"RAGAS context precision (with reference) = {score_ref:.3f}")

/tmp/claude-501/ipykernel_75676/1971641820.py:1: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import LLMContextPrecisionWithReference


RAGAS context precision (with reference) = 1.000
